In [2]:
import tensorflow as tf
import tensorflow.keras as keras
from keras import layers
import deeplake as dl
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

c:\Users\etito\miniconda3\Lib\site-packages\deeplake\util\check_latest_version.py:32: UserWarning: A newer version of deeplake (4.6.3) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(


In [3]:
# Loading the Dataset
train_ds = dl.load("hub://activeloop/visdrone-det-train")
val_ds = dl.load("hub://activeloop/visdrone-det-val")
test_ds = dl.load("hub://activeloop/visdrone-det-test-dev")
print(train_ds.summary())

Opening dataset in read-only mode as you don't have write permissions.


/

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/visdrone-det-train



|

hub://activeloop/visdrone-det-train loaded successfully.



...

Opening dataset in read-only mode as you don't have write permissions.


/

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/visdrone-det-test-dev



|

hub://activeloop/visdrone-det-test-dev loaded successfully.

Dataset(path='hub://activeloop/visdrone-det-train', read_only=True, tensors=['boxes', 'images', 'labels'])

 tensor      htype                 shape               dtype  compression
 -------    -------               -------             -------  ------- 
  boxes      bbox            (6471, 1:914, 4)         float32   None   
 images      image     (6471, 360:1500, 480:2000, 3)   uint8    jpeg   
 labels   class_label          (6471, 1:914)          uint32    None   
None


In [17]:
# Variables
BATCH_SIZE = 128
AUTOTUNE = tf.data.AUTOTUNE

GLOBAL_BACKBONE = keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3)
)
GLOBAL_BACKBONE.trainable = False

city_labels = ["tianjin", "hong-kong", "daqing", "ganzhou", 
               "guangzhou", "jinchang", "liuzhou", "nanjing", 
               "shaoxing", "shenyang", "nanyang", "zhangjiakou", 
               "suzhou", "xuzhou"]

In [ ]:
# Preprocessing Detection

def image_preprocessing(image):
    image = tf.image.resize(image, (224, 224))
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image

def detection_preprocessing(data):
    # Extracts the image and calculates separate counts for humans and cars.
    image = data['images']
    labels = data['labels']  # Integer Class IDs from VisDrone (0 to 9)
    
    # 1. Process the image
    image = image_preprocessing(image)
    
    # 2. Track Humans: Pedestrians (0) OR People (1)
    is_human_mask = tf.logical_or(tf.equal(labels, 0), tf.equal(labels, 1))
    human_count = tf.reduce_sum(tf.cast(is_human_mask, tf.float32))
    
    # 3. Track Cars: Car (3)
    is_car_mask = tf.equal(labels, 3)
    car_count = tf.reduce_sum(tf.cast(is_car_mask, tf.float32))
    
    # 4. Combine into a single multi-count vector: [human_count, car_count]
    # Squeezing elements into a 1D tensor of shape [2]
    counts_vector = tf.stack([human_count, car_count], axis=0)

    # 5. Normalize counts
    normalized_counts = counts_vector / 100
    
    return image, normalized_counts

# Feature Stream Pipelines

train_stream = (train_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))

val_stream = (val_ds.tensorflow()
                .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                .batch(BATCH_SIZE)
                .prefetch(AUTOTUNE))

test_stream = (test_ds.tensorflow()
                 .map(detection_preprocessing, num_parallel_calls=AUTOTUNE)
                 .batch(BATCH_SIZE)
                 .prefetch(AUTOTUNE))

In [23]:
# Preprocessing City Labels

def classification_preprocessing(data_stream, is_training, pca_model=None, kmeans_model=None):
    features = GLOBAL_BACKBONE.predict(data_stream, verbose=0)

    if is_training:
        # Create and fit the models
        pca_model = PCA(n_components=50, random_state=42)
        kmeans_model = MiniBatchKMeans(
            n_clusters=14,
            random_state=42,
            n_init=5,
            batch_size=8192
        )
        reduced_features = pca_model.fit_transform(features)
        labels = kmeans_model.fit_predict(reduced_features)
        
        # CRITICAL: Return the labels AND the trained models
        return tf.constant(labels, dtype=tf.int32), pca_model, kmeans_model

    else:
        # Safety check: make sure the caller actually passed the trained models
        if pca_model is None or kmeans_model is None:
            raise ValueError("You must pass the trained pca_model and kmeans_model when is_training=False")
            
        # Use the models passed from the outside
        reduced_features = pca_model.transform(features)
        labels = kmeans_model.predict(reduced_features)
        
        # For inference, just return the labels
        return tf.constant(labels, dtype=tf.int32)

# Training phase: captures the trained PCA and KMeans weights
tf_train_labels, trained_pca, trained_kmeans = classification_preprocessing(
    train_stream.map(lambda img, _ : img), is_training=True
)

# Validation and Test phases: reuse those exact trained weights
tf_val_labels = classification_preprocessing(
    val_stream.map(lambda img, _ : img), is_training=False, pca_model=trained_pca, kmeans_model=trained_kmeans
)
tf_test_labels = classification_preprocessing(
    test_stream.map(lambda img, _ : img), is_training=False, pca_model=trained_pca, kmeans_model=trained_kmeans
)

train_labels_ds = tf.data.Dataset.from_tensor_slices(tf_train_labels).batch(BATCH_SIZE)
val_labels_ds = tf.data.Dataset.from_tensor_slices(tf_val_labels).batch(BATCH_SIZE)
test_labels_ds = tf.data.Dataset.from_tensor_slices(tf_test_labels).batch(BATCH_SIZE)

c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [24]:
# Adding Pipelines

def shape_structure_outputs(img_batch, count_batch, label_batch):
    # Enforce static ranks for the Keras graph builder
    img_batch.set_shape([None, 224, 224, 3])
    count_batch.set_shape([None, 2])
    label_batch.set_shape([None])   # [Batch_Size]
    
    return img_batch, {
        "city_output": label_batch,
        "count_output": count_batch
    }

# Zip your feature streams (images + counts) with your clustering labels
train_pipeline = tf.data.Dataset.zip((train_stream, train_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_pipeline = tf.data.Dataset.zip((val_stream, val_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

test_pipeline = tf.data.Dataset.zip((test_stream, test_labels_ds)).map(
    lambda stream_data, labels: shape_structure_outputs(stream_data[0], stream_data[1], labels),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

In [28]:
# Building the Model
def build_multi_head_model():
    input_tensor = layers.Input(shape=(224, 224, 3), name='input_image')

    base_model = keras.applications.MobileNetV2(
        weights='imagenet', 
        include_top=False, 
        input_tensor=input_tensor
    )
    base_model.trainable = False

    shared_features = base_model.output

    # 1. Object Detection Head (Processes backbone features first)
    x_detect = layers.GlobalAveragePooling2D()(shared_features)
    x_detect = layers.Dense(128, activation='relu')(x_detect)
    count_output = layers.Dense(2, activation='relu', name="count_output")(x_detect)

    # 2. City Classification Head (Dependent on Detection)
    x_class_input = layers.GlobalAveragePooling2D()(shared_features)
    
    # Concatenate blends the spatial context with the localization features
    merged_features = layers.Concatenate()([x_class_input, x_detect])
    
    x_class = layers.Dense(256, activation='relu')(merged_features)
    x_class = layers.Dropout(0.5)(x_class)
    city_output = layers.Dense(14, activation='softmax', name="city_output")(x_class)

    model = keras.Model(inputs=input_tensor, outputs=[city_output, count_output])

    lr_scheduler = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=0.001,
        decay_steps=10000,
        alpha=0.0
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr_scheduler),
        loss={
            "city_output": "sparse_categorical_crossentropy",
            "count_output": "mean_absolute_error"
        },
        metrics={
            "city_output": "accuracy",
            "count_output": "mae"
        },
        loss_weights={
            "city_output": 1.0, 
            "count_output": 0.1
        }
    )
    model.fit(
        train_pipeline,
        validation_data=val_pipeline,
        epochs=5
    )
    
    return model

china_model = build_multi_head_model()

C:\Users\etito\AppData\Local\Temp\ipykernel_1748\160069920.py:5: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(


Epoch 1/5
     51/Unknown 225s 4s/step - city_output_accuracy: 0.4590 - city_output_loss: 1.6736 - count_output_loss: 0.0911 - count_output_mae: 0.0911 - loss: 1.6828

c:\Users\etito\miniconda3\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


51/51 ━━━━━━━━━━━━━━━━━━━━ 242s 5s/step - city_output_accuracy: 0.6090 - city_output_loss: 1.1114 - count_output_loss: 0.0786 - count_output_mae: 0.0803 - loss: 1.1464 - val_city_output_accuracy: 0.8285 - val_city_output_loss: 0.3869 - val_count_output_loss: 0.0856 - val_count_output_mae: 0.1052 - val_loss: 0.4841
Epoch 2/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 240s 5s/step - city_output_accuracy: 0.8110 - city_output_loss: 0.5067 - count_output_loss: 0.0749 - count_output_mae: 0.0765 - loss: 0.5266 - val_city_output_accuracy: 0.8686 - val_city_output_loss: 0.3082 - val_count_output_loss: 0.0855 - val_count_output_mae: 0.1050 - val_loss: 0.3802
Epoch 3/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 221s 4s/step - city_output_accuracy: 0.8486 - city_output_loss: 0.3993 - count_output_loss: 0.0747 - count_output_mae: 0.0763 - loss: 0.4160 - val_city_output_accuracy: 0.8704 - val_city_output_loss: 0.2964 - val_count_output_loss: 0.0855 - val_count_output_mae: 0.1050 - val_loss: 0.3781
Epoch 4/5
51/51 ━━━━━━━━━━━━━━━━

In [29]:
# Testing Model

results = china_model.evaluate(test_pipeline)

metrics_names = china_model.metrics_names
print("\nTest Metrics Breakdown:")
for name, value in zip(metrics_names, results):
    print(f"- {name}: {value:.4f}")

for test_images, test_targets in test_pipeline.take(1):
    
    # Generate predictions for this batch
    city_preds, count_preds = china_model.predict(test_images)
    
    # Extract the highest probability index
    predicted_city_indices = np.argmax(city_preds, axis=-1)
    true_city_indices = test_targets["city_output"].numpy().flatten()
    true_counts = test_targets["count_output"].numpy()
    
    print("\nBatch Sample Breakdown:")
    for i in range(len(test_images)):
        # Map integer indices back to string labels
        pred_label = city_labels[predicted_city_indices[i]]
        true_label = city_labels[true_city_indices[i]]
        
        print(f"\nSample {i+1}:")
        print(f"  [City] Predicted: {pred_label:<12} | True: {true_label}")
        print(f"  [Count] Predicted: {count_preds[i]*100} | True: {true_counts[i]*100}")

13/13 ━━━━━━━━━━━━━━━━━━━━ 48s 4s/step - city_output_accuracy: 0.8807 - city_output_loss: 0.2766 - count_output_loss: 0.0704 - count_output_mae: 0.0761 - loss: 0.3057

Test Metrics Breakdown:
- loss: 0.3057
- compile_metrics: 0.2766
- city_output_loss: 0.0704
- count_output_loss: 0.8807
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 462ms/step

Batch Sample Breakdown:

Sample 1:
  [City] Predicted: nanyang      | True: nanjing
  [Count] Predicted: [0. 0.] | True: [0. 0.]

Sample 2:
  [City] Predicted: nanjing      | True: hong-kong
  [Count] Predicted: [0. 0.] | True: [0. 0.]

Sample 3:
  [City] Predicted: suzhou       | True: hong-kong
  [Count] Predicted: [0. 0.] | True: [11. 29.]

Sample 4:
  [City] Predicted: nanyang      | True: nanyang
  [Count] Predicted: [0. 0.] | True: [10.  0.]

Sample 5:
  [City] Predicted: guangzhou    | True: guangzhou
  [Count] Predicted: [0. 0.] | True: [0. 0.]

Sample 6:
  [City] Predicted: liuzhou      | True: liuzhou
  [Count] Predicted: [0. 0.] | True: [6. 2.]

Sample 